#  Plankton-Klassifizierung für ML
---

## 1. Motivation & Zielsetzung

### **Motivation:**
- **Praktische Erfahrung:** Erste Anwendung bildbasierter Machine-Learning-Methoden in einem Projekt.
- **Unbalancierte Datensätze:** Umgang mit stark unbalancierten Klassen, was bisher im Studium kaum behandelt wurde.
- **Realistische Problemstellungen:** Arbeit mit echten Herausforderungen der Datenklassifikation statt idealisierten Szenarien.

---

### **Zielsetzungen:**
- **Vergleich der Ansätze:**
  - Entwicklung klassischer ML-Modelle ( Decison Trees und KNN ).
  - Aufbau und Einsatz neuronaler Netze (Feedforward Neural Networks und Convolutional Neural Networks).
  
<hr style="border: 1px solid #ccc; margin: 20px 0;" />

## 2. Datensatzbeschreibung
---

### Art der Daten:
| Eigenschaft        | Details                                                                      |
|--------------------|-----------------------------------------------------------------------------|
| **Format**         | PNG-Bildformat.                                                            |
| **Größe**          | Unterschiedliche Bildgrößen ohne einheitliche Skalierung.                  |
| **Farbformat**     | Schwarz-, Grau- und Weißtöne; einige Bilder enthalten grüne Farbtöne.      |


---

### **Labels & Klasseninformation:**
- Die Klassifikation der Bilder erfolgt ausschließlich anhand des Dateinamens, der die Plankton-Art eindeutig ausweist.

---

### Umfang des Datensatzes:
| Eigenschaft  | Details            |
|--------------|--------------------|
| **Anzahl**   | 219.063 Bilder.    |
| **Größe**    | ca. 8 GB.          |
---

### **Empfohlene datenbezogene Ordnerstruktur:**
- Das Projekt enthält eine klar definierte Ordnerstruktur, um die Bilder übersichtlich zu verwalten:
  - **Bilderordner:** Alle Bilder werden zentral in `./data/Plankton-Bilder` gespeichert.
  - **Beispielstruktur:** 
    ```
    .
      ── /data
          ── /Plankton-Bilder
              ── fs446_ap...100.tif.png
              ── fs446_ap...300.tif.png
              ── ...
    ```
- Diese Struktur erleichtert die Nachvollziehbarkeit und lokale Verarbeitung.

---

### **Technischer Hinweis:**
- Die bereitgestellten 8 GB des Datensatzes stellen für lokale Maschinen hohe Speicheransprüche.
- Nur Teilmengen werden für Entwicklungs- und Testzwecke genutzt, um die Verarbeitung effizienter zu gestalten.

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


In [ ]:
import os
import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import re

# Dynamischer Pfad basierend auf Benutzerverzeichnis
base_path = os.path.expanduser("../data/Plankton-Bilder") 

# Überprüfen, ob der Pfad existiert
if not os.path.exists(base_path):
    raise FileNotFoundError(f"Der angegebene Pfad existiert nicht: {base_path}")

print(f"Datenpfad erkannt: {base_path}")

# Unterstützte Dateiformate
supported_formats = (".png", ".jpg", ".jpeg", ".tiff")

# Funktion zur dynamischen Extraktion von Labels aus Dateinamen
def extract_label_from_filename(filename):
    match = re.search(r"_(.*?)_roi", filename)
    return match.group(1) if match else "Unknown"

# Zum Sammeln von Metadaten
image_metadata = []

In [ ]:
# Übersicht über den Datensatz 
print(" Lade und analysiere Daten...")

# Informationen aller unterstützten Dateien sammeln
for filename in os.listdir(base_path):
    if filename.lower().endswith(supported_formats):
        # Dynamische Label-Extraktion
        label = extract_label_from_filename(filename)
        file_path = os.path.join(base_path, filename)

        # Bild öffnen und Metadaten extrahieren
        with Image.open(file_path) as img:
            width, height = img.size
            color_mode = "RGB" if len(img.getbands()) == 3 else "Grayscale"

        # Metadata speichern
        image_metadata.append({
            "filename": filename,
            "label": label,
            "width": width,
            "height": height,
            "color_mode": color_mode
        })

# DataFrame erstellen
df = pd.DataFrame(image_metadata)

# Berechnung der Gesamtspeichergröße
total_size = sum(os.path.getsize(os.path.join(base_path, img)) for img in df["filename"])
print(f"Gesamtspeichergröße des Datensatzes: {total_size / (1024 ** 3):.2f} GB")

# Erste Analyse
print("Datensatz geladen.")
print("Anzahl der Bilder:", len(df))
print("Labels:", df["label"].unique())
print("-" * 50)

# Anzeige von 3 Beispielbildern 
def show_example_images(df, base_path, num_images=3):
    
    # Zufällige Auswahl von Bildern
    random_images = random.sample(df["filename"].tolist(), num_images)

    plt.figure(figsize=(15, 5))
    for i, img_name in enumerate(random_images):
        file_path = os.path.join(base_path, img_name)
        with Image.open(file_path) as img:
            plt.subplot(1, num_images, i+1)
            plt.imshow(img, cmap="gray" if img.mode != "RGB" else None)
            plt.title(f"Label: {extract_label_from_filename(img_name)}")
            plt.axis("off")

    plt.tight_layout()
    plt.show()

# Beispielbilder anzeigen
print(" Zeige 3 Beispielbilder aus dem Datensatz...")
show_example_images(df, base_path)

In [ ]:
# Klassenverteilung 
print("Klassenverteilung:")
label_counts = df["label"].value_counts()
print(label_counts)

# Klassenverteilung visualisieren
plt.figure(figsize=(12, 6))
sns.barplot(x=label_counts.index, y=label_counts.values, palette="viridis")
plt.title("Verteilung der Klassen")
plt.xlabel("Klassen")
plt.ylabel("Anzahl der Bilder")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Bildgrößen und Aspekte 
print(" Analyse der Bildgrößen...")
print(" Breite: Min =", df["width"].min(), ", Max =", df["width"].max())
print(" Höhe: Min =", df["height"].min(), ", Max =", df["height"].max())

In [ ]:
# Farbmodusanalyse 
color_mode_counts = df["color_mode"].value_counts()
print("\nFarbmodi:")
print(color_mode_counts)

In [ ]:
# Zusammenfassung 
print("\n Zusammenfassung der Erkenntnisse:")
print(f"- Insgesamt {len(df)} Bilder im Datensatz.")
print("- Klassenlabels:", df['label'].unique())
print("- Speichergröße: {:.2f} GB.".format(total_size / (1024 ** 3)))
print("  Breite: Min =", df["width"].min(), ", Max =", df["width"].max())
print("  Höhe: Min =", df["height"].min(), ", Max =", df["height"].max())
print(color_mode_counts.to_string())
print("\n EDA abgeschlossen!")

# **Zusammenfassung der Ergebnisse**

## **1. Allgemeine Eigenschaften des Datensatzes**
- **Gesamtdaten:**
  - Insgesamt **219.062 Bilder** im Datensatz.
  - **Speichergröße:** 7.64 GB.
- **Bildgrößen:**
  - **Breite:** Min = 19 Pixel, Max = 1379 Pixel.
  - **Höhe:** Min = 17 Pixel, Max = 1379 Pixel.
  - Die Bildgrößen variieren 
- **Farbmodus:**
  - Alle Bilder sind im **RGB-Farbmodus**.

---

## **2. Klassenbeschreibung**
- **Labels:**
  - Insgesamt gibt es **30 Klassen**:
    `snow`, `pluteus`, `pteropods`, `copepods`, `diatoms`, `noctiluca`, `worms`, `appendicularia`, usw.
  - Die Bilder sind teilweise uneindeutig deshalb die Klassen (`unknown`, `blurry`).

- **Klassenverteilung:**
  - **Unbalanced:** Extrem viele Bilder in der Klasse `snow` (**143.295 Bilder, ~65% des Datensatzes**).
  - Die zweitgrößte Klasse, `pluteus`, enthält nur **28.974 Bilder** (~13%). Die restlichen Klassen sind deutlich kleiner: 
    - Beispiele: 
      - `actinotrocha`: **264 Bilder**
      - `pilidium`: **244 Bilder**

<hr style="border: 1px solid #ccc; margin: 20px 0;" />

## **Probleme mit dem Datensatz**


### **1. Starke Klassenunbalanciertheit**
- Die dominierende Klasse `snow` (65%) führt zu einer Verzerrung beim Training, da das Modell dazu neigt, diese Klasse zu bevorzugen.
- Kleine Klassen wie `pilidium` oder `echinodermata` sind unterrepräsentiert und könnten von Modellen schlecht generalisiert werden.

### **2. Varianz der Bildgröße**
- Breite und Höhe variieren stark (von 19×17 bis 1379×1379 Pixel).

### **3. Uneindeutige Labels**
- Klassen wie `unknown` und `blurry` tragen wenig zur Modellleistung bei.

### **4. Ähnlichkeit der Klassen**
- Einige Klassen (z. B. `zoea`, `pluteus`, `larvae`) sind sehr ähnlich und könnten durch ein Modell schwer zu unterscheiden sein.

### **5. Bildqualität**
- Einige Bilder könnten unscharf sein (`blurry`) oder weniger relevante Informationen enthalten.

### **6. Speicher- und Skalierungsprobleme**
- Der große Datenumfang (7.64 GB) könnte auf Speicherbeschränkungen stoßen und zu erhöhter Laufzeit führen.

<hr style="border: 1px solid #ccc; margin: 20px 0;" />


## **Problemlösung**

### **1. Umgang mit starker Klassenunbalanciertheit**
- Nach Rücksprache habe wir entschieden, uns auf die **5 am häufigsten vertretenen Klassen** zu konzentrieren. 
- Jede dieser Klassen wird mit **5.000 bis 10.000 Bildern** repräsentiert, um eine ausreichende Datenbasis zu gewährleisten und die Klassenunbalanciertheit effektiv zu verringern.

### **2. Vorverarbeitung der Bilder**
- Einheitliches **Resizing** auf eine standardisierte Größe (z. B. 224×224 Pixel), um eine konsistente Eingabedimension für die Modelle zu schaffen.
- **Farbvorkomprimierung**, falls Speicherengpässe auftreten, um die Daten handhabbarer zu machen.

### **3. Label-Überprüfung**
- Durch die Fokussierung auf die **5 am häufigsten vorkommenden Klassen** entfällt das Problem, da diese Klassen eindeutig unterscheidbar sind.

### **4. Feature-Extraktion**
- Dieses Problem entfällt ebenso, aufgrund der Begrenzung auf die Top 5 Klassen. Diese sind eindeutig unterscheidbar voneinander 

### **5. Datenbereinigung**
- Durch die Fokussierung auf die **5 am häufigsten vorkommenden Klassen** entfällt das Problem

### **6. Speicher- und Skalierungsprobleme**
- Durch die Fokussierung auf die **5 am häufigsten vorkommenden Klassen** und begrenzung auf **5.000 bis 10.000 Bildern** entfällt das Problem.

<hr style="border: 1px solid #ccc; margin: 20px 0;" />

# Mit diesem Vorgehen wird die Qualität des Datensatzes verbessert und die Performance der Machine-Learning-Modelle erheblich optimiert.

<hr style="border: 1px solid #ccc; margin: 20px 0;" />